In [152]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

In [153]:
data = 'https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv'
!wget $data -O data_hw.csv 

--2026-06-04 11:54:12--  https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 80876 (79K) [text/plain]
Saving to: ‘data_hw.csv’

data_hw.csv         100%[===================>]  78.98K  --.-KB/s    in 0.002s  

2026-06-04 11:54:12 (49.9 MB/s) - ‘data_hw.csv’ saved [80876/80876]



In [154]:
df = pd.read_csv('data_hw.csv')
df.head()

,lead_source,industry,number_of_courses_viewed,annual_income,employment_status,location,interaction_count,lead_score,converted
0,paid_ads,NaN,1,79450.0,unemployed,south_america,4,0.94,1
1,social_media,retail,1,46992.0,employed,south_america,1,0.80,0
2,events,healthcare,5,78796.0,unemployed,australia,3,0.69,1
3,paid_ads,retail,2,83843.0,NaN,australia,1,0.87,0
4,referral,education,3,85012.0,self_employed,europe,3,0.62,1


## Data preparation
Check if the missing values are presented in the features.
If there are missing values:
1. For categorical features, replace them with 'NA'
2. For numerical features, replace with with 0.0

In [155]:
df.isnull().sum()

lead_source                 128
industry                    134
number_of_courses_viewed      0
annual_income               181
employment_status           100
location                     63
interaction_count             0
lead_score                    0
converted                     0
dtype: int64

In [156]:
categorical = ['lead_source', 'industry', 'employment_status', 'location']
numerical = ['number_of_courses_viewed', 'annual_income', 'interaction_count', 'lead_score']
df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)

In [157]:
df.head()

,lead_source,industry,number_of_courses_viewed,annual_income,employment_status,location,interaction_count,lead_score,converted
0,paid_ads,NA,1,79450.0,unemployed,south_america,4,0.94,1
1,social_media,retail,1,46992.0,employed,south_america,1,0.80,0
2,events,healthcare,5,78796.0,unemployed,australia,3,0.69,1
3,paid_ads,retail,2,83843.0,NA,australia,1,0.87,0
4,referral,education,3,85012.0,self_employed,europe,3,0.62,1


Q1. What is the most frequent observation (mode) for the column industry? d. retail

In [158]:
df.industry.mode()

0    retail
Name: industry, dtype: str

Q2. What are the two features that have the biggest correlation?
- interaction_count and lead_score
- number_of_courses_viewed and lead_score
- number_of_courses_viewed and interaction_count
- annual_income and interaction_count

Only consider the pairs above when answering this question. d. annual_income and interaction_count

In [159]:
from sklearn.metrics import mutual_info_score

In [160]:
df[numerical].columns

Index(['number_of_courses_viewed', 'annual_income', 'interaction_count',
       'lead_score'],
      dtype='str')

In [161]:
A = np.zeros((len(df[numerical].columns), len(df[numerical].columns)))
i = 0
for col in df[numerical].columns:
    j = 0
    for row in df[numerical].columns:
        A[i, j] = mutual_info_score(df[col], df[row])
        j += 1
    i += 1
A

/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/metrics/cluster/_supervised.py:68: UserWarning: Clustering metrics expects discrete values but received multiclass values for label, and continuous values for target
  warnings.warn(msg, UserWarning)
/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/metrics/cluster/_supervised.py:68: UserWarning: Clustering metrics expects discrete values but received multiclass values for label, and continuous values for target
  warnings.warn(msg, UserWarning)
/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/metrics/cluster/_supervised.py:68: UserWarning: Clustering metrics expects discrete values but received multiclass values for label, and continuous values for target
  warnings.warn(msg, UserWarning)
/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/metrics/cluster/_supervised.py:68: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and multiclass valu

array([[1.71452134, 1.49068   , 0.02533981, 0.22982933],
       [1.49068   , 6.63069601, 1.65557343, 4.04548707],
       [0.02533981, 1.65557343, 1.90460733, 0.28943459],
       [0.22982933, 4.04548707, 0.28943459, 4.58182076]])

## Split the data
1. Split your data in train/val/test sets with 60%/20%/20% distribution.
2. Use Scikit-Learn for that (the train_test_split function) and set the seed to 42.
3. Make sure that the target value converted is not in your dataframe.

In [162]:
from sklearn.model_selection import train_test_split

In [163]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

df_full_train = df_full_train.reset_index(drop=True)
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_full_train = df_full_train.converted.values
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_full_train['converted']
del df_train['converted']
del df_val['converted']
del df_test['converted']

In [164]:
len(df_train), len(df_val), len(df_test)

(876, 293, 293)

Q3. Calculate the mutual information score between converted and other categorical variables in the dataset. Use the training set only.
Round the scores to 2 decimals using round(score, 2).
Which of these variables has the biggest mutual information score? c. lead_source
- industry
- location
- lead_source
- employment_status

In [165]:
def mutual_info_score_series(series):
    return mutual_info_score(series, pd.Series(y_train))
mi = df_train[categorical].apply(mutual_info_score_series)
mi.sort_values(ascending=False)

lead_source          0.035396
employment_status    0.012938
industry             0.011575
location             0.004464
dtype: float64

Q4. Now let's train a logistic regression.
Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
Fit the model on the training dataset.
To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
Calculate the accuracy on the validation dataset and round it to 2 decimal digits.
What accuracy did you get? b. 0.74

In [166]:
from sklearn.feature_extraction import DictVectorizer

In [167]:
dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

In [168]:
from sklearn.linear_model import LogisticRegression

In [169]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_val)
ori_acc = (y_pred == y_val).mean()
ori_acc

np.float64(0.6996587030716723)

Q5. Let's find the least useful feature using the feature elimination technique.
Train a model using the same features and parameters as in Q4 (without rounding).
Now exclude each feature from this set and train a model without it. Record the accuracy for each model.
For each feature, calculate the difference between the original accuracy and the accuracy without the feature.
Which of following feature has the smallest difference? a. 'industry'

- 'industry'
- 'employment_status'
- 'lead_score'

In [170]:
to_remove = ['industry', 'employment_status', 'lead_score']
for col in to_remove:
    features = categorical + numerical
    print(col)
    features.remove(col)
    
    dv = DictVectorizer(sparse=False)

    train_dict = df_train[features].to_dict(orient='records')
    X_train = dv.fit_transform(train_dict)
    
    val_dict = df_val[features].to_dict(orient='records')
    X_val = dv.transform(val_dict)

    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    acc = (y_pred == y_val).mean()
    print(acc - ori_acc, acc)

industry
0.0 0.6996587030716723
employment_status
-0.0034129692832763903 0.6962457337883959
lead_score
0.0068259385665528916 0.7064846416382252


Q6. Now let's train a regularized logistic regression.
Let's try the following values of the parameter C: [0.01, 0.1, 1, 10, 100].
Train models using all the features as in Q4.
Calculate the accuracy on the validation dataset and round it to 3 decimal digits.
Which of these C leads to the best accuracy on the validation set? a. 0.01

In [174]:
C = [0.01, 0.1, 1, 10, 100]
for c in C:
    dv = DictVectorizer(sparse=False)

    train_dict = df_train[categorical + numerical].to_dict(orient='records')
    X_train = dv.fit_transform(train_dict)
    
    val_dict = df_val[categorical + numerical].to_dict(orient='records')
    X_val = dv.transform(val_dict)

    model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    acc = (y_pred == y_val).mean()
    print(c, acc)

0.01 0.6996587030716723
0.1 0.6996587030716723
1 0.6996587030716723
10 0.6996587030716723
100 0.6996587030716723
